In [ ]:
# BAD: Crowded stacked bar that mixes unrelated concepts
import pandas as pd
import plotly.express as px
df = pd.read_csv("WHR2024.csv")
factors = ["Explained by: Log GDP per capita", "Explained by: Social support", "Explained by: Healthy life expectancy"]
df_long = df.melt(id_vars=["Country name"], value_vars=factors,
                  var_name="Factor", value_name="Contribution")
fig = px.bar(df_long, x="Country name", y="Contribution", color="Factor",
             title="Happiness Factor Contributions by Country (Bad Example)")
fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("WHR2024.csv")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

df_2024 = df[df["Year"] == 2024]
top20 = df_2024.sort_values("Ladder score", ascending=False).head(10)

fig = px.bar(
    top20,
    x="Ladder score",
    y="Country name",
    orientation="h",
    title="Top 10 Countries by Happiness Score (2024)",
    labels={"Ladder score": "Happiness Score", "Country name": "Country"},
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


In [ ]:
# BAD: Adds a speculative color dimension with no justification
fig = px.scatter(
    df_2024,
    x="Explained by: Log GDP per capita",
    y="Ladder score",
    color="Explained by: Perceptions of corruption",  # introduces unrelated variable
    title="GDP vs Happiness with Corruption (Speculative Overlay)"
)
fig.show()


In [ ]:
# BETTER: Clean scatter with linear fit
import numpy as np
import plotly.graph_objects as go

df_gdp = df_2024.dropna(subset=["Explained by: Log GDP per capita", "Ladder score"])

x = df_gdp["Explained by: Log GDP per capita"]
y = df_gdp["Ladder score"]
a, b = np.polyfit(x, y, 1)

fig = px.scatter(
    df_gdp,
    x="Explained by: Log GDP per capita",
    y="Ladder score",
    hover_name="Country name",
    title="GDP vs Happiness Score (2024)"
)

xline = np.linspace(x.min(), x.max(), 100)
yline = a * xline + b
fig.add_trace(go.Scatter(x=xline, y=yline, mode="lines", name="Linear fit"))
fig.show()


In [ ]:
# BAD: Pie chart of components that vary per country, shown as global average
drivers = [
    "Explained by: Log GDP per capita",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Generosity",
    "Explained by: Perceptions of corruption"
]

avg = df_2024[drivers].mean()

fig = px.pie(
    names=drivers,
    values=avg,
    title="Average Global Happiness Contributions (Misleading)"
)
fig.show()


In [ ]:
# BETTER: Just show how Social Support varies
col = "Explained by: Social support"
df_valid = df_2024[df_2024[col].notna()]

fig = px.histogram(
    df_valid,
    x=col,
    nbins=20,
    title="Distribution of Social Support Contributions to Happiness (2024)",
    labels={col: "Contribution to Ladder Score"},
)
fig.update_layout(yaxis_title="Number of Countries")
fig.show()
